# NeuroGolf Solver Family: Fill / Additive Marking - nonlocal_1color

This notebook builds the first scoped submission for the `fill_enclosed_regions / nonlocal_1color` subtype.

Workflow:

1. Load the strict 41-task subtype from `task_type_map.csv`.
2. Try bounded wider additive-template models for tasks where 3x3 context is insufficient.
3. Fall back to compact additive candidates when they visibly fit.
4. Emit identity fallback models for any remaining task so the submission zip is complete.
5. Build `/kaggle/working/submission.zip` from exactly this subtype scope.

The generated maps are heuristic solver-routing labels. Validate against visible examples before submitting.

In [1]:
# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-nonlocal-1color-v0.5-correctness"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})




def make_initializer(name, array):
    arr = np.asarray(array, dtype=np.float16)
    return numpy_helper.from_array(arr, name=name)

def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path# Inline helper functions from submission_nbs/neurogolf_nb_common.py
"""Shared helpers for NeuroGolf submission notebooks.

The notebooks in this folder are solver-family workbooks. They should be
copied into Kaggle or run locally with the competition files available.
"""

import json
import math
import os
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import onnx
    import onnxruntime as ort
    from onnx import TensorProto, helper, numpy_helper
except Exception:  # Notebook analysis cells can still run without ONNX.
    onnx = None
    ort = None
    TensorProto = None
    helper = None
    numpy_helper = None


BATCH, CH, H, W = 1, 10, 30, 30
MODEL_VERSION = "fill-additive-nonlocal-1color-v0.5-correctness"


def default_paths():
    kaggle_dir = Path("/kaggle/input/competitions/neurogolf-2026")
    if kaggle_dir.exists():
        data_dir = kaggle_dir
        root = Path("/kaggle/working")
    else:
        root = Path.cwd()
        data_dir = root / "competition_material" / "taskfiles"
        if not data_dir.exists():
            data_dir = root / "competition_material"
    out_dir = root / "working_submission"
    out_dir.mkdir(parents=True, exist_ok=True)
    return data_dir, out_dir


def load_task_type_map(path="/kaggle/input/datasets/prince22466/task-type-map-csv/task_type_map.csv"):
    import pandas as pd

    candidates = [Path(path), Path("task_groups/task_type_map.csv")]
    for candidate in candidates:
        if candidate.exists():
            return pd.read_csv(candidate, dtype={"task_id": str})
    raise FileNotFoundError(f"task_type_map.csv not found in: {candidates}")


def load_task_groups(path="task_groups/task_type_groups.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def family_task_ids(family, groups_path="/kaggle/input/datasets/prince22466/task-type-groups-json/task_type_groups.json"):
    groups_candidates = [Path(groups_path), Path("task_groups/task_type_groups.json")]
    for candidate in groups_candidates:
        if candidate.exists():
            groups = load_task_groups(candidate)
            return groups.get(family, [])
    raise FileNotFoundError(f"task_type_groups.json not found in: {groups_candidates}")


def task_num(task_id):
    return int(str(task_id).replace("task", ""))


def task_path(data_dir, task_id):
    data_dir = Path(data_dir)
    name = f"{task_id}.json" if str(task_id).startswith("task") else f"task{int(task_id):03d}.json"
    direct = data_dir / name
    if direct.exists():
        return direct
    nested = data_dir / "taskfiles" / name
    if nested.exists():
        return nested
    raise FileNotFoundError(name)


def load_task(data_dir, task_id):
    with task_path(data_dir, task_id).open("r", encoding="utf-8") as f:
        return json.load(f)


def all_examples(task):
    return task.get("train", []) + task.get("test", []) + task.get("arc-gen", [])


def grid_shape(grid):
    return len(grid), len(grid[0]) if grid else 0


def grid_to_tensor(grid):
    arr = np.zeros((BATCH, CH, H, W), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if 0 <= r < H and 0 <= c < W:
                arr[0, int(color), r, c] = 1.0
    return arr


def tensor_to_grid(arr):
    arr = np.asarray(arr)
    if arr.ndim == 4:
        arr = arr[0]
    grid = []
    for r in range(H):
        row = []
        for c in range(W):
            vals = np.where(arr[:, r, c] > 0.5)[0]
            row.append(int(vals[0]) if len(vals) == 1 else 0)
        while row and row[-1] == 0:
            row.pop()
        grid.append(row)
    while grid and not grid[-1]:
        grid.pop()
    return grid


def require_onnx():
    if onnx is None or helper is None or TensorProto is None or numpy_helper is None:
        raise ImportError("onnx is required to build models")


def make_model(nodes, initializers, opset=10):
    require_onnx()
    # Keep competition-facing graph I/O as float32. Cast internally to float16.
    inp = helper.make_tensor_value_info("input", TensorProto.FLOAT, [BATCH, CH, H, W])
    out = helper.make_tensor_value_info("output", TensorProto.FLOAT, [BATCH, CH, H, W])
    for node in nodes:
        for i, value in enumerate(node.input):
            if value == "input":
                node.input[i] = "input_f16"
        for i, value in enumerate(node.output):
            if value == "output":
                node.output[i] = "output_f16"
    cast_in = helper.make_node("Cast", ["input"], ["input_f16"], to=TensorProto.FLOAT16)
    cast_out = helper.make_node("Cast", ["output_f16"], ["output"], to=TensorProto.FLOAT)
    graph = helper.make_graph([cast_in] + list(nodes) + [cast_out], "graph", [inp], [out], initializers)
    return helper.make_model(graph, ir_version=10, opset_imports=[helper.make_opsetid("", opset)])


def make_identity_model():
    return make_1x1_color_model({c: c for c in range(CH)})


def make_1x1_color_model(mapping):
    require_onnx()
    dt = TensorProto.FLOAT16
    weights = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    bias = np.full((CH,), -0.5, dtype=np.float16)
    for ic in range(CH):
        oc = int(mapping.get(ic, ic))
        weights[oc, ic, 0, 0] = 1.0
    w = make_initializer("W", weights)
    b = make_initializer("B", bias)
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"], kernel_shape=[1, 1])
    return make_model([node], [w, b])


def infer_global_color_mapping(examples):
    mapping = {}
    for ex in examples:
        inp, out = ex["input"], ex["output"]
        if grid_shape(inp) != grid_shape(out):
            return None
        for r, row in enumerate(inp):
            for c, ic in enumerate(row):
                oc = out[r][c]
                prev = mapping.get(int(ic))
                if prev is None:
                    mapping[int(ic)] = int(oc)
                elif prev != int(oc):
                    return None
    for c in range(CH):
        mapping.setdefault(c, c)
    return mapping


def train_color_remap_model(task):
    mapping = infer_global_color_mapping(all_examples(task))
    if mapping is None:
        return None, {"ok": False, "reason": "no consistent global color mapping"}
    return make_1x1_color_model(mapping), {"ok": True, "mapping": mapping}


def fixed_transform(grid, transform):
    arr = np.array(grid, dtype=int)
    if transform == "rot90":
        return np.rot90(arr, -1).tolist()
    if transform == "rot180":
        return np.rot90(arr, 2).tolist()
    if transform == "rot270":
        return np.rot90(arr, 1).tolist()
    if transform == "flip_h":
        return np.fliplr(arr).tolist()
    if transform == "flip_v":
        return np.flipud(arr).tolist()
    if transform == "transpose":
        return arr.T.tolist()
    raise ValueError(transform)


def infer_fixed_geometric_transform(examples):
    names = ["rot90", "rot180", "rot270", "flip_h", "flip_v", "transpose"]
    matches = []
    for name in names:
        if all(fixed_transform(ex["input"], name) == ex["output"] for ex in examples):
            matches.append(name)
    return matches


def save_model(model, out_dir, task_id):
    require_onnx()
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"{task_id}.onnx"
    onnx.save(model, path)
    return path


def run_model(model_or_path, input_grid):
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required to run validation")
    if isinstance(model_or_path, (str, Path)):
        session = ort.InferenceSession(str(model_or_path), providers=["CPUExecutionProvider"])
    else:
        session = ort.InferenceSession(model_or_path.SerializeToString(), providers=["CPUExecutionProvider"])
    output = session.run(["output"], {"input": grid_to_tensor(input_grid)})[0]
    return (output > 0).astype(np.float32)


def visible_validation_summary(model_or_path, task, max_examples=None):
    examples = all_examples(task)
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    return {"right": right, "wrong": wrong, "first_wrong": first_wrong}


def split_examples(task):
    return {
        "train": task.get("train", []),
        "test": task.get("test", []),
        "arc_gen": task.get("arc-gen", []),
    }


def validation_summary_for_examples(model_or_path, examples, max_examples=None):
    if max_examples is not None:
        examples = examples[:max_examples]
    right = 0
    wrong = 0
    first_wrong = None
    for ex in examples:
        expected = grid_to_tensor(ex["output"])
        actual = run_model(model_or_path, ex["input"])
        if np.array_equal(actual, expected):
            right += 1
        else:
            wrong += 1
            if first_wrong is None:
                first_wrong = ex
    total = right + wrong
    accuracy = right / total if total else None
    return {"right": right, "wrong": wrong, "total": total, "accuracy": accuracy, "first_wrong": first_wrong}


def split_validation_summary(model_or_path, task, max_examples=None):
    rows = {}
    for split, examples in split_examples(task).items():
        summary = validation_summary_for_examples(model_or_path, examples, max_examples=max_examples)
        summary.pop("first_wrong", None)
        rows[split] = summary
    visible = validation_summary_for_examples(model_or_path, all_examples(task), max_examples=max_examples)
    visible.pop("first_wrong", None)
    rows["visible_all"] = visible
    return rows


def count_model_params(model_or_path):
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    params = 0
    for init in model.graph.initializer:
        if init.dims:
            params += math.prod(init.dims)
        else:
            params += 1
    for node in model.graph.node:
        if node.op_type != "Constant":
            continue
        for attr in node.attribute:
            if attr.name == "value":
                params += math.prod(attr.t.dims) if attr.t.dims else 1
            elif attr.name == "value_floats":
                params += len(attr.floats)
            elif attr.name == "value_ints":
                params += len(attr.ints)
            elif attr.name == "value_strings":
                params += len(attr.strings)
    return int(params)


def model_architecture_summary(model_or_path):
    require_onnx()
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    op_counts = Counter(node.op_type for node in model.graph.node)
    init_shapes = {init.name: list(init.dims) for init in model.graph.initializer}
    return {
        "ir_version": model.ir_version,
        "opsets": {op.domain or "ai.onnx": op.version for op in model.opset_import},
        "nodes": len(model.graph.node),
        "op_counts": dict(op_counts),
        "initializers": init_shapes,
        "params": count_model_params(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
    }


def approximate_memory_from_model_shapes(model_or_path):
    """Approximate scored tensor memory from static value_info shapes.

    The official helper uses ONNX Runtime profiling to refine tensor memory.
    This approximation is useful in notebooks before running the full profiler.
    """
    require_onnx()
    model = onnx.load(str(model_or_path)) if isinstance(model_or_path, (str, Path)) else model_or_path
    inferred = onnx.shape_inference.infer_shapes(model)
    graph = inferred.graph
    total = 0
    for value in list(graph.value_info):
        tensor_type = value.type.tensor_type
        if not tensor_type.HasField("shape"):
            continue
        dims = []
        for dim in tensor_type.shape.dim:
            if not dim.HasField("dim_value") or dim.dim_value <= 0:
                dims = []
                break
            dims.append(dim.dim_value)
        if dims:
            total += math.prod(dims) * 2
    return int(total)


def runtime_memory_profile(model_or_path, sample_input_grid):
    """Return an approximate competition memory/param profile.

    This uses ONNX Runtime profiling if available. It is not a replacement for
    the official Kaggle validator, but it tracks the same concerns: parameters,
    intermediate tensor memory, and file size.
    """
    require_onnx()
    if ort is None:
        raise ImportError("onnxruntime is required for runtime memory profiling")
    path = Path(model_or_path) if isinstance(model_or_path, (str, Path)) else None
    model = onnx.load(str(path)) if path else model_or_path
    options = ort.SessionOptions()
    options.enable_profiling = True
    options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_DISABLE_ALL
    session = ort.InferenceSession(model.SerializeToString(), options, providers=["CPUExecutionProvider"])
    session.run(["output"], {"input": grid_to_tensor(sample_input_grid)})
    trace_path = session.end_profiling()
    trace_memory = 0
    try:
        with open(trace_path, "r", encoding="utf-8") as f:
            trace = json.load(f)
        for event in trace:
            args = event.get("args", {})
            for shape_dict in args.get("output_type_shape", []) or []:
                for dims in shape_dict.values():
                    if dims and all(isinstance(d, int) and d > 0 for d in dims):
                        trace_memory += math.prod(dims) * 2
    except Exception:
        trace_memory = approximate_memory_from_model_shapes(model)
    return {
        "params": count_model_params(model),
        "runtime_memory_bytes": int(trace_memory),
        "static_memory_bytes": approximate_memory_from_model_shapes(model),
        "file_size_bytes": path.stat().st_size if path and path.exists() else None,
        "profile_trace_path": trace_path,
    }


def model_report(model_or_path, task=None, sample_input_grid=None, max_validation_examples=None):
    report = {"architecture": model_architecture_summary(model_or_path)}
    if task is not None:
        report["performance"] = split_validation_summary(
            model_or_path,
            task,
            max_examples=max_validation_examples,
        )
        if sample_input_grid is None:
            examples = all_examples(task)
            if examples:
                sample_input_grid = examples[0]["input"]
    if sample_input_grid is not None and ort is not None:
        report["memory_profile"] = runtime_memory_profile(model_or_path, sample_input_grid)
    else:
        report["memory_profile"] = {
            "params": report["architecture"]["params"],
            "static_memory_bytes": approximate_memory_from_model_shapes(model_or_path),
            "runtime_memory_bytes": None,
            "file_size_bytes": report["architecture"]["file_size_bytes"],
            "profile_trace_path": None,
        }
    return report


def create_submission_zip(model_dir, zip_path=None):
    model_dir = Path(model_dir)
    if zip_path is None:
        zip_path = model_dir / "submission.zip"
    else:
        zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(model_dir.glob("task*.onnx")):
            zf.write(path, path.name)
    return zip_path


def build_family_submission(family, trainer, data_dir, out_dir, fallback_identity=False, validate=False, task_ids_override=None):
    task_ids = list(task_ids_override) if task_ids_override is not None else family_task_ids(family)
    rows = []
    for task_id in task_ids:
        task = load_task(data_dir, task_id)
        model, info = trainer(task)
        if model is None and fallback_identity:
            model = make_identity_model()
            info = {**info, "fallback": "identity"}
        if model is None:
            rows.append({"task_id": task_id, "saved": False, **info})
            continue
        path = save_model(model, out_dir, task_id)
        row = {"task_id": task_id, "saved": True, "path": str(path), **info}
        if validate:
            try:
                row.update({f"visible_{k}": v for k, v in visible_validation_summary(path, task).items() if k != "first_wrong"})
            except Exception as exc:
                row["visible_error"] = repr(exc)
        rows.append(row)
    zip_path = create_submission_zip(out_dir)
    return rows, zip_path


In [2]:
from pathlib import Path
import ast
import json
import pandas as pd

ROOT = Path.cwd()

FAMILY = 'fill_enclosed_regions'
SUBTYPE = 'nonlocal_1color'
MODEL_VERSION = 'fill-additive-nonlocal-1color-v0.5-correctness'
DATA_DIR, BASE_OUT_DIR = default_paths()
OUT_DIR = BASE_OUT_DIR / f'{FAMILY}_{SUBTYPE}'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('OUT_DIR =', OUT_DIR)
print('MODEL_VERSION =', MODEL_VERSION)


DATA_DIR = /kaggle/input/competitions/neurogolf-2026
OUT_DIR = /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color
MODEL_VERSION = fill-additive-nonlocal-1color-v0.5-correctness


In [3]:
# ONNX dependency setup for model export.
# Dry-run rule fitting can run without ONNX, but build_family_submission
# must import onnx to create taskNNN.onnx files.
import importlib.util
import subprocess
import sys

missing = [pkg for pkg in ['onnx', 'onnxruntime'] if importlib.util.find_spec(pkg) is None]
if missing:
    print('Installing missing ONNX packages:', missing)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import onnx
import onnxruntime as ort
from onnx import TensorProto, helper, numpy_helper

print('onnx:', onnx.__version__)
print('onnxruntime:', ort.__version__)

Installing missing ONNX packages: ['onnxruntime']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 70.8 MB/s eta 0:00:00
onnx: 1.20.1
onnxruntime: 1.27.0


In [4]:
task_map = load_task_type_map()
family_df = task_map[task_map.primary_family == FAMILY].copy()


def parse_color_list(value):
    if pd.isna(value) or value == '':
        return []
    if isinstance(value, list):
        return value
    try:
        return list(ast.literal_eval(str(value)))
    except Exception:
        return []

family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
nonlocal_1color_df = family_df[
    family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
    & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
    & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
].copy()
nonlocal_1color_df = nonlocal_1color_df.sort_values('task_id').reset_index(drop=True)
task_ids = nonlocal_1color_df['task_id'].tolist()

print('family:', FAMILY)
print('subtype:', SUBTYPE)
print('family tasks:', len(family_df))
print('selected nonlocal_1color tasks:', len(task_ids))
print(task_ids)
display(nonlocal_1color_df.head(10))


family: fill_enclosed_regions
subtype: nonlocal_1color
family tasks: 59
selected nonlocal_1color tasks: 41
['task002', 'task027', 'task042', 'task043', 'task047', 'task050', 'task060', 'task063', 'task090', 'task102', 'task105', 'task119', 'task126', 'task139', 'task162', 'task166', 'task176', 'task200', 'task219', 'task232', 'task246', 'task251', 'task255', 'task265', 'task273', 'task278', 'task299', 'task303', 'task323', 'task335', 'task336', 'task341', 'task348', 'task350', 'task357', 'task367', 'task371', 'task381', 'task387', 'task392', 'task397']


,task_id,task_num,primary_family,confidence,candidate_flags,n_train,n_test,n_arc_gen,n_examples,shape_relation,...,mapping_conflicts,fixed_geometric_transforms,local_3x3_score,local_3x3_conflicts,local_3x3_samples,input_nonzero_preserved_ratio,added_nonzero_cells,changed_cells,notes,parsed_new_output_colors
0,task002,2,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,5,1,262,268,same_shape_variable_size,...,9828,NaN,0.919284,1421,17605,1.0,9828,9828,Same shape; input is mostly preserved while ne...,[4]
1,task027,27,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,261,265,same_shape_variable_size,...,1260,NaN,0.951500,291,6000,1.0,1260,1260,Same shape; input is mostly preserved while ne...,[2]
2,task042,42,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,1071,NaN,0.948667,308,6000,1.0,1071,1071,Same shape; input is mostly preserved while ne...,[8]
3,task043,43,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,5392,NaN,0.799167,1205,6000,1.0,5392,5392,Same shape; input is mostly preserved while ne...,[2]
4,task047,47,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,2,1,262,265,same_shape_variable_size,...,7950,NaN,0.718930,1366,4860,1.0,7950,7950,Same shape; input is mostly preserved while ne...,[2]
5,task050,50,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,8,1,262,271,same_shape_variable_size,...,1289,NaN,0.922996,365,4740,1.0,1289,1289,Same shape; input is mostly preserved while ne...,[3]
6,task060,60,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,2,1,262,265,same_shape_variable_size,...,4095,NaN,0.792121,686,3300,1.0,4095,4095,Same shape; input is mostly preserved while ne...,[5]
7,task063,63,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,3,1,262,266,same_shape_variable_size,...,8094,NaN,0.777271,2001,8984,1.0,8094,8094,Same shape; input is mostly preserved while ne...,[3]
8,task090,90,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,4,1,262,267,same_shape_variable_size,...,2984,NaN,0.909534,483,5339,1.0,2984,2984,Same shape; input is mostly preserved while ne...,[6]
9,task102,102,fill_enclosed_regions,medium,adds_new_color_preserves_input|new_output_colors,4,1,262,267,same_shape_variable_size,...,978,NaN,0.950463,428,8640,1.0,978,978,Same shape; input is mostly preserved while ne...,[2]


In [5]:
# Inspect one scoped task quickly.
if task_ids:
    sample_task_id = task_ids[0]
    sample_task = load_task(DATA_DIR, sample_task_id)
    print(sample_task_id, 'examples:', len(all_examples(sample_task)))
    print('first input shape:', grid_shape(sample_task['train'][0]['input']))
    print('first output shape:', grid_shape(sample_task['train'][0]['output']))
    print('first input:', sample_task['train'][0]['input'])
    print('first output:', sample_task['train'][0]['output'])
else:
    print('No tasks currently mapped to this subtype.')


task002 examples: 268
first input shape: (6, 6)
first output shape: (6, 6)
first input: [[0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0], [0, 3, 0, 3, 0, 0], [0, 0, 3, 0, 3, 0], [0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0]]
first output: [[0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0], [0, 3, 4, 3, 0, 0], [0, 0, 3, 4, 3, 0], [0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0]]


In [6]:
# nonlocal_1color selection table: these are the tasks this notebook version is responsible for.
selection_cols = [
    'task_id',
    'confidence',
    'n_train',
    'n_test',
    'n_arc_gen',
    'shape_relation',
    'input_shape_modes',
    'output_shape_modes',
    'input_color_list',
    'output_color_list',
    'new_output_color_list',
    'local_3x3_score',
    'local_3x3_conflicts',
    'added_nonzero_cells',
    'candidate_flags',
]
fill_selection = nonlocal_1color_df[selection_cols].reset_index(drop=True)
print('selected nonlocal_1color fill/additive tasks:', len(fill_selection))
display(fill_selection)


selected nonlocal_1color fill/additive tasks: 41


,task_id,confidence,n_train,n_test,n_arc_gen,shape_relation,input_shape_modes,output_shape_modes,input_color_list,output_color_list,new_output_color_list,local_3x3_score,local_3x3_conflicts,added_nonzero_cells,candidate_flags
0,task002,medium,5,1,262,same_shape_variable_size,20x20:75;19x19:40;18x18:39;16x16:21;15x15:20,20x20:75;19x19:40;18x18:39;16x16:21;15x15:20,"[0,3]","[0,3,4]",[4],0.919284,1421,9828,adds_new_color_preserves_input|new_output_colors
1,task027,medium,3,1,261,same_shape_variable_size,10x10:265,10x10:265,"[0,1]","[0,1,2]",[2],0.951500,291,1260,adds_new_color_preserves_input|new_output_colors
2,task042,medium,3,1,262,same_shape_variable_size,10x10:266,10x10:266,"[0,3]","[0,3,8]",[8],0.948667,308,1071,adds_new_color_preserves_input|new_output_colors
3,task043,medium,3,1,262,same_shape_variable_size,10x10:266,10x10:266,"[0,5]","[0,2,5]",[2],0.799167,1205,5392,adds_new_color_preserves_input|new_output_colors
4,task047,medium,2,1,262,same_shape_variable_size,9x9:265,9x9:265,"[0,7,8]","[0,2,7,8]",[2],0.718930,1366,7950,adds_new_color_preserves_input|new_output_colors
5,task050,medium,8,1,262,same_shape_variable_size,14x7:6;11x5:6;10x4:5;8x12:5;3x5:5,14x7:6;11x5:6;10x4:5;8x12:5;3x5:5,"[0,8]","[0,3,8]",[3],0.922996,365,1289,adds_new_color_preserves_input|new_output_colors
6,task060,medium,2,1,262,same_shape_variable_size,5x11:265,5x11:265,"[0,1,2,3,4,6,7,8,9]","[0,1,2,3,4,5,6,7,8,9]",[5],0.792121,686,4095,adds_new_color_preserves_input|new_output_colors
7,task063,medium,3,1,262,same_shape_variable_size,14x14:93;10x10:87;12x12:86,14x14:93;10x10:87;12x12:86,"[0,2,8]","[0,2,3,8]",[3],0.777271,2001,8094,adds_new_color_preserves_input|new_output_colors
8,task090,medium,4,1,262,same_shape_variable_size,4x21:13;2x20:11;4x29:11;4x24:10;4x22:10,4x21:13;2x20:11;4x29:11;4x24:10;4x22:10,"[0,1,5]","[0,1,5,6]",[6],0.909534,483,2984,adds_new_color_preserves_input|new_output_colors
9,task102,medium,4,1,262,same_shape_variable_size,12x12:267,12x12:267,"[0,5]","[0,2,5]",[2],0.950463,428,978,adds_new_color_preserves_input|new_output_colors


In [7]:
# Minimal nonlocal_1color v4 trainer: correctness-first semantic models plus identity fallback.
# task050: connect matching color-8 endpoints horizontally or vertically with color 3.
# task126: detect U-shapes of the form xxx/x0x and mark the bottom in-grid row under each center column with color 4.
# task299: extend the horizontal color-2 line and vertical color-8 line; intersection becomes color 4.
# task357: fill the variable-width grid with 8 and draw the bouncing diagonal color-1 path.
# task176: complete a fixed 3-row periodic color-4 pattern.
# task371: pure simulator only for midpoint-plus correctness tracking; ONNX export is intentionally deferred.

CLEAR = 10
ZERO_HOT = -1
NO_CHANGE = -2


def input_canvas(grid):
    canvas = np.full((H, W), CLEAR, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def output_canvas(grid):
    canvas = np.full((H, W), ZERO_HOT, dtype=np.int16)
    for r, row in enumerate(grid):
        for c, color in enumerate(row):
            if r < H and c < W:
                canvas[r, c] = int(color)
    return canvas


def task_color_sets(task):
    in_colors = sorted({int(v) for ex in all_examples(task) for row in ex['input'] for v in row})
    out_colors = sorted({int(v) for ex in all_examples(task) for row in ex['output'] for v in row})
    return in_colors, out_colors


def exact_grid_match(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    for split in ['train', 'test', 'arc-gen']:
        for idx, ex in enumerate(task.get(split, [])):
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                right += 1
            elif first_wrong is None:
                first_wrong = (split, idx)
    return right, total, first_wrong


def simulate_task050_line_connect(input_grid):
    marker = 8
    fill = 3
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    pred = [list(row) for row in input_grid]
    coords = [(r, c) for r, row in enumerate(input_grid) for c, value in enumerate(row) if int(value) == marker]
    for idx, (r1, c1) in enumerate(coords):
        for r2, c2 in coords[idx + 1:]:
            if r1 == r2:
                left, right = sorted((c1, c2))
                if all(int(input_grid[r1][c]) == 0 for c in range(left + 1, right)):
                    for c in range(left + 1, right):
                        pred[r1][c] = fill
            if c1 == c2:
                top, bottom = sorted((r1, r2))
                if all(int(input_grid[r][c1]) == 0 for r in range(top + 1, bottom)):
                    for r in range(top + 1, bottom):
                        pred[r][c1] = fill
    return pred


def simulate_task126_u_bottom(input_grid):
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    pred = [list(row) for row in input_grid]
    for r in range(h - 1):
        for c in range(w - 2):
            color = int(input_grid[r][c])
            if color == 0:
                continue
            if (
                int(input_grid[r][c + 1]) == color
                and int(input_grid[r][c + 2]) == color
                and int(input_grid[r + 1][c]) == color
                and int(input_grid[r + 1][c + 1]) == 0
                and int(input_grid[r + 1][c + 2]) == color
            ):
                pred[h - 1][c + 1] = 4
    return pred


def fit_task050_line_connect(task):
    in_colors, out_colors = task_color_sets(task)
    if in_colors != [0, 8] or out_colors != [0, 3, 8]:
        return None, {
            'ok': False,
            'trainer': 'task050_line_connect_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not color-8 line connector',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task050_line_connect)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task050_line_connect_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'line-connect simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task050_line_connect'}, {
        'ok': True,
        'trainer': 'task050_line_connect_cnn',
        'model_version': MODEL_VERSION,
        'reason': None,
        'semantic_kind': 'connect_color8_pairs_row_or_column_with_3',
        'visible_right': right,
        'visible_total': total,
        'estimated_params': 1500,
        'estimated_static_memory_bytes': 30 * 30 * 16,
    }


def fit_task126_u_bottom(task):
    in_colors, out_colors = task_color_sets(task)
    if 4 in in_colors or 4 not in out_colors:
        return None, {
            'ok': False,
            'trainer': 'task126_u_bottom_marker_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not U-bottom marker',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task126_u_bottom)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task126_u_bottom_marker_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'U-bottom simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task126_u_bottom_marker'}, {
        'ok': True,
        'trainer': 'task126_u_bottom_marker_cnn',
        'model_version': MODEL_VERSION,
        'reason': None,
        'semantic_kind': 'mark_bottom_row_under_u_shape_centers',
        'visible_right': right,
        'visible_total': total,
        'estimated_params': 2500,
        'estimated_static_memory_bytes': 30 * 30 * 16,
    }


def simulate_task299_cross_extend(input_grid):
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    pred = [list(row) for row in input_grid]
    cols = [c for c in range(w) if sum(1 for r in range(h) if int(input_grid[r][c]) == 8) >= 2]
    rows = [r for r in range(h) if sum(1 for c in range(w) if int(input_grid[r][c]) == 2) >= 2]
    for c in cols:
        for r in range(h):
            pred[r][c] = 8
    for r in rows:
        for c in range(w):
            pred[r][c] = 2
    for r in rows:
        for c in cols:
            pred[r][c] = 4
    return pred


def bounce_col(step, width):
    if width <= 1:
        return 0
    period = 2 * (width - 1)
    value = step % period
    return value if value < width else period - value


def simulate_task357_bounce_path(input_grid):
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    seeds = [(r, c) for r, row in enumerate(input_grid) for c, value in enumerate(row) if int(value) == 1]
    if not seeds:
        return [list(row) for row in input_grid]
    seed_row, seed_col = seeds[0]
    pred = []
    for r in range(h):
        row = []
        for c in range(w):
            step = seed_row - r
            path_col = seed_col + bounce_col(step, w)
            row.append(1 if c == path_col else 8)
        pred.append(row)
    return pred


def simulate_task176_periodic_completion(input_grid):
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    pred = [list(row) for row in input_grid]
    if h != 3:
        return pred
    for c in range(w):
        if c % 12 in (5, 6, 7) and int(pred[0][c]) == 0:
            pred[0][c] = 4
        if c % 6 == 0 and int(pred[1][c]) == 0:
            pred[1][c] = 4
        if c % 12 in (0, 1, 11) and int(pred[2][c]) == 0:
            pred[2][c] = 4
    return pred


def simulate_task371_midpoint_plus(input_grid):
    h = len(input_grid)
    w = len(input_grid[0]) if h else 0
    pred = [list(row) for row in input_grid]
    points = [(r, c) for r, row in enumerate(input_grid) for c, value in enumerate(row) if int(value) == 1]
    if len(points) != 2:
        return pred
    (r1, c1), (r2, c2) = points
    if r1 != r2 and c1 != c2:
        return pred
    mid_r = (r1 + r2) // 2
    mid_c = (c1 + c2) // 2
    for rr, cc in [(mid_r, mid_c), (mid_r - 1, mid_c), (mid_r + 1, mid_c), (mid_r, mid_c - 1), (mid_r, mid_c + 1)]:
        if 0 <= rr < h and 0 <= cc < w and int(pred[rr][cc]) == 0:
            pred[rr][cc] = 3
    return pred


def fit_task299_cross_extend(task):
    in_colors, out_colors = task_color_sets(task)
    shapes = sorted({(len(ex['input']), len(ex['input'][0])) for ex in all_examples(task)})
    if in_colors != [0, 2, 8] or out_colors != [0, 2, 4, 8] or shapes != [(6, 6)]:
        return None, {
            'ok': False,
            'trainer': 'task299_cross_extend_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not 6x6 2/8 cross extension',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task299_cross_extend)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task299_cross_extend_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'cross-extension simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task299_cross_extend'}, {
        'ok': True,
        'trainer': 'task299_cross_extend_cnn',
        'model_version': MODEL_VERSION,
        'reason': None,
        'semantic_kind': 'extend_2_row_and_8_column_with_4_intersection',
        'visible_right': right,
        'visible_total': total,
        'estimated_params': 20000,
        'estimated_static_memory_bytes': 30 * 30 * 32,
    }


def fit_task357_bounce_path(task):
    in_colors, out_colors = task_color_sets(task)
    shapes = sorted({(len(ex['input']), len(ex['input'][0])) for ex in all_examples(task)})
    widths = sorted({w for h, w in shapes})
    if in_colors != [0, 1] or out_colors != [1, 8] or any(h != 10 for h, _w in shapes) or min(widths) < 2 or max(widths) > 10:
        return None, {
            'ok': False,
            'trainer': 'task357_bounce_path_gemm',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not 10xh bounce path',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task357_bounce_path)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task357_bounce_path_gemm',
            'model_version': MODEL_VERSION,
            'reason': 'bounce-path simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task357_bounce_path', 'widths': widths}, {
        'ok': True,
        'trainer': 'task357_bounce_path_gemm',
        'model_version': MODEL_VERSION,
        'reason': None,
        'semantic_kind': 'variable_width_bouncing_diagonal_path',
        'visible_right': right,
        'visible_total': total,
        'estimated_params': 9000 * len(widths),
        'estimated_static_memory_bytes': 30 * 30 * 32,
    }


def fit_task176_periodic_completion(task):
    in_colors, out_colors = task_color_sets(task)
    shapes = sorted({(len(ex['input']), len(ex['input'][0])) for ex in all_examples(task)})
    if in_colors != [0, 2] or out_colors != [0, 2, 4] or any(h != 3 for h, _w in shapes):
        return None, {
            'ok': False,
            'trainer': 'task176_periodic_completion_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not 3-row periodic completion',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task176_periodic_completion)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task176_periodic_completion_cnn',
            'model_version': MODEL_VERSION,
            'reason': 'periodic-completion simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task176_periodic_completion'}, {
        'ok': True,
        'trainer': 'task176_periodic_completion_cnn',
        'model_version': MODEL_VERSION,
        'reason': None,
        'semantic_kind': '3_row_periodic_color4_completion',
        'visible_right': right,
        'visible_total': total,
        'estimated_params': 250,
        'estimated_static_memory_bytes': 30 * 30 * 8,
    }


def fit_task371_midpoint_plus_simulator(task):
    in_colors, out_colors = task_color_sets(task)
    if in_colors != [0, 1] or out_colors != [0, 1, 3]:
        return None, {
            'ok': False,
            'trainer': 'task371_midpoint_plus_simulator_only',
            'model_version': MODEL_VERSION,
            'reason': 'task signature is not two-endpoint midpoint plus',
        }
    right, total, first_wrong = exact_grid_match(task, simulate_task371_midpoint_plus)
    if right != total:
        return None, {
            'ok': False,
            'trainer': 'task371_midpoint_plus_simulator_only',
            'model_version': MODEL_VERSION,
            'reason': 'midpoint-plus simulation failed visible examples',
            'visible_right': right,
            'visible_total': total,
            'first_wrong': first_wrong,
        }
    return {'kind': 'task371_midpoint_plus_simulator_only'}, {
        'ok': True,
        'trainer': 'task371_midpoint_plus_simulator_only',
        'model_version': MODEL_VERSION,
        'reason': 'visible-perfect simulator; ONNX export deferred',
        'semantic_kind': 'two_endpoint_midpoint_plus_marker',
        'visible_right': right,
        'visible_total': total,
        'export_status': 'simulator_only',
    }


def make_base_identity_initializers():
    W_base = np.zeros((CH, CH, 1, 1), dtype=np.float16)
    B_base = np.full((CH,), -0.5, dtype=np.float16)
    for color in range(CH):
        W_base[color, color, 0, 0] = 1.0
    return make_initializer('W_base', W_base), make_initializer('B_base', B_base)


def make_zero_mask_initializer(name='W_zero_mask'):
    W_zero = np.zeros((1, CH, 1, 1), dtype=np.float16)
    B_zero = np.asarray([-0.5], dtype=np.float16)
    W_zero[0, 0, 0, 0] = 1.0
    return make_initializer(name, W_zero), make_initializer('B_zero_mask', B_zero)


def make_marker_delta_initializer(out_color, name_w='W_marker_delta', name_b='B_marker_delta'):
    W_delta = np.zeros((CH, 1, 1, 1), dtype=np.float16)
    B_delta = np.zeros((CH,), dtype=np.float16)
    W_delta[int(out_color), 0, 0, 0] = 8.0
    W_delta[0, 0, 0, 0] = -8.0
    return make_initializer(name_w, W_delta), make_initializer(name_b, B_delta)


def make_task050_line_connect_model(payload):
    require_onnx()
    W_base, B_base = make_base_identity_initializers()
    W_zero, B_zero = make_zero_mask_initializer()

    W_left = np.zeros((1, CH, 1, 30), dtype=np.float16)
    W_left[0, 8, 0, :] = 1.0
    B_has = np.asarray([-0.5], dtype=np.float16)
    W_up = np.zeros((1, CH, 30, 1), dtype=np.float16)
    W_up[0, 8, :, 0] = 1.0
    W_delta, B_delta = make_marker_delta_initializer(3)

    initializers = [
        W_base, B_base, W_zero, B_zero,
        make_initializer('W_left', W_left), make_initializer('B_left', B_has),
        make_initializer('W_right', W_left), make_initializer('B_right', B_has),
        make_initializer('W_up', W_up), make_initializer('B_up', B_has),
        make_initializer('W_down', W_up), make_initializer('B_down', B_has),
        W_delta, B_delta,
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_zero_mask', 'B_zero_mask'], ['zero_logits'], kernel_shape=[1, 1]),
        helper.make_node('Relu', ['zero_logits'], ['zero_mask']),
        helper.make_node('Conv', ['input', 'W_left', 'B_left'], ['left_logits'], kernel_shape=[1, 30], pads=[0, 29, 0, 0]),
        helper.make_node('Conv', ['input', 'W_right', 'B_right'], ['right_logits'], kernel_shape=[1, 30], pads=[0, 0, 0, 29]),
        helper.make_node('Conv', ['input', 'W_up', 'B_up'], ['up_logits'], kernel_shape=[30, 1], pads=[29, 0, 0, 0]),
        helper.make_node('Conv', ['input', 'W_down', 'B_down'], ['down_logits'], kernel_shape=[30, 1], pads=[0, 0, 29, 0]),
        helper.make_node('Relu', ['left_logits'], ['left_has']),
        helper.make_node('Relu', ['right_logits'], ['right_has']),
        helper.make_node('Relu', ['up_logits'], ['up_has']),
        helper.make_node('Relu', ['down_logits'], ['down_has']),
        helper.make_node('Mul', ['left_has', 'right_has'], ['horizontal_between']),
        helper.make_node('Mul', ['up_has', 'down_has'], ['vertical_between']),
        helper.make_node('Add', ['horizontal_between', 'vertical_between'], ['between_any']),
        helper.make_node('Mul', ['between_any', 'zero_mask'], ['marker_hits']),
        helper.make_node('Conv', ['marker_hits', 'W_marker_delta', 'B_marker_delta'], ['marker_logits'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'marker_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)


def make_task126_u_bottom_model(payload):
    require_onnx()
    W_base, B_base = make_base_identity_initializers()

    colors = list(range(1, 10))
    W_u = np.zeros((len(colors), CH, 3, 3), dtype=np.float16)
    B_u = np.full((len(colors),), -5.5, dtype=np.float16)
    expected_positions = [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2)]
    for out_idx, color in enumerate(colors):
        for kr, kc in expected_positions:
            W_u[out_idx, :, kr, kc] = -1.0
            expected_color = 0 if (kr, kc) == (1, 1) else color
            W_u[out_idx, expected_color, kr, kc] = 1.0

    W_sum = np.ones((1, len(colors), 1, 1), dtype=np.float16)
    B_sum = np.zeros((1,), dtype=np.float16)

    W_project = np.ones((1, 1, 30, 1), dtype=np.float16)
    B_project = np.asarray([-0.25], dtype=np.float16)

    W_bottom = np.zeros((1, CH, 2, 1), dtype=np.float16)
    B_bottom = np.asarray([-0.5], dtype=np.float16)
    W_bottom[0, 0, 0, 0] = 1.0
    W_bottom[0, :, 1, 0] = -1.0

    W_delta, B_delta = make_marker_delta_initializer(4)
    initializers = [
        W_base, B_base,
        make_initializer('W_u_shape', W_u), make_initializer('B_u_shape', B_u),
        make_initializer('W_u_sum', W_sum), make_initializer('B_u_sum', B_sum),
        make_initializer('W_project_col', W_project), make_initializer('B_project_col', B_project),
        make_initializer('W_bottom_mask', W_bottom), make_initializer('B_bottom_mask', B_bottom),
        W_delta, B_delta,
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_u_shape', 'B_u_shape'], ['u_logits'], kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
        helper.make_node('Relu', ['u_logits'], ['u_hits_by_color']),
        helper.make_node('Conv', ['u_hits_by_color', 'W_u_sum', 'B_u_sum'], ['u_hits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['u_hits', 'W_project_col', 'B_project_col'], ['project_logits'], kernel_shape=[30, 1], pads=[29, 0, 0, 0]),
        helper.make_node('Relu', ['project_logits'], ['project_has']),
        helper.make_node('Conv', ['input', 'W_bottom_mask', 'B_bottom_mask'], ['bottom_logits'], kernel_shape=[2, 1], pads=[0, 0, 1, 0]),
        helper.make_node('Relu', ['bottom_logits'], ['bottom_mask']),
        helper.make_node('Mul', ['project_has', 'bottom_mask'], ['marker_hits']),
        helper.make_node('Conv', ['marker_hits', 'W_marker_delta', 'B_marker_delta'], ['marker_logits'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'marker_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)


def make_task299_cross_extend_model(payload):
    require_onnx()
    W_base, B_base = make_base_identity_initializers()
    flat_size = CH * H * W
    spatial_size = H * W

    # Six row detectors for rows with at least two color-2 cells in the 6x6 grid.
    W_row = np.zeros((6, flat_size), dtype=np.float16)
    B_row = np.full((6,), -1.5, dtype=np.float16)
    for r in range(6):
        for c in range(6):
            W_row[r, 2 * H * W + r * W + c] = 1.0

    # Six column detectors for columns with at least two color-8 cells in the 6x6 grid.
    W_col = np.zeros((6, flat_size), dtype=np.float16)
    B_col = np.full((6,), -1.5, dtype=np.float16)
    for c in range(6):
        for r in range(6):
            W_col[c, 8 * H * W + r * W + c] = 1.0

    W_row_grid = np.zeros((6, spatial_size), dtype=np.float16)
    W_col_grid = np.zeros((6, spatial_size), dtype=np.float16)
    for r in range(6):
        for c in range(6):
            W_row_grid[r, r * W + c] = 1.0
            W_col_grid[c, r * W + c] = 1.0

    W_row_delta = np.zeros((CH, 1, 1, 1), dtype=np.float16)
    B_row_delta = np.zeros((CH,), dtype=np.float16)
    W_row_delta[2, 0, 0, 0] = 8.0
    W_row_delta[0, 0, 0, 0] = -8.0

    W_col_delta = np.zeros((CH, 1, 1, 1), dtype=np.float16)
    B_col_delta = np.zeros((CH,), dtype=np.float16)
    W_col_delta[8, 0, 0, 0] = 8.0
    W_col_delta[0, 0, 0, 0] = -8.0

    W_inter_delta = np.zeros((CH, 1, 1, 1), dtype=np.float16)
    B_inter_delta = np.zeros((CH,), dtype=np.float16)
    W_inter_delta[4, 0, 0, 0] = 16.0
    W_inter_delta[2, 0, 0, 0] = -16.0
    W_inter_delta[8, 0, 0, 0] = -16.0
    W_inter_delta[0, 0, 0, 0] = -16.0

    shape_1_1_hw = numpy_helper.from_array(np.asarray([1, 1, H, W], dtype=np.int64), 'shape_1_1_hw')
    initializers = [
        W_base, B_base,
        numpy_helper.from_array(W_row, 'W_row_det'), numpy_helper.from_array(B_row, 'B_row_det'),
        numpy_helper.from_array(W_col, 'W_col_det'), numpy_helper.from_array(B_col, 'B_col_det'),
        numpy_helper.from_array(W_row_grid, 'W_row_grid'),
        numpy_helper.from_array(W_col_grid, 'W_col_grid'),
        shape_1_1_hw,
        make_initializer('W_row_delta', W_row_delta), make_initializer('B_row_delta', B_row_delta),
        make_initializer('W_col_delta', W_col_delta), make_initializer('B_col_delta', B_col_delta),
        make_initializer('W_inter_delta', W_inter_delta), make_initializer('B_inter_delta', B_inter_delta),
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Flatten', ['input'], ['flat_input'], axis=1),
        helper.make_node('Gemm', ['flat_input', 'W_row_det', 'B_row_det'], ['row_logits'], transB=1),
        helper.make_node('Gemm', ['flat_input', 'W_col_det', 'B_col_det'], ['col_logits'], transB=1),
        helper.make_node('Relu', ['row_logits'], ['row_hits']),
        helper.make_node('Relu', ['col_logits'], ['col_hits']),
        helper.make_node('Gemm', ['row_hits', 'W_row_grid'], ['row_grid_flat']),
        helper.make_node('Gemm', ['col_hits', 'W_col_grid'], ['col_grid_flat']),
        helper.make_node('Reshape', ['row_grid_flat', 'shape_1_1_hw'], ['row_grid']),
        helper.make_node('Reshape', ['col_grid_flat', 'shape_1_1_hw'], ['col_grid']),
        helper.make_node('Mul', ['row_grid', 'col_grid'], ['inter_grid']),
        helper.make_node('Conv', ['row_grid', 'W_row_delta', 'B_row_delta'], ['row_logits_out'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['col_grid', 'W_col_delta', 'B_col_delta'], ['col_logits_out'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['inter_grid', 'W_inter_delta', 'B_inter_delta'], ['inter_logits_out'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'row_logits_out'], ['with_row']),
        helper.make_node('Add', ['with_row', 'col_logits_out'], ['with_col']),
        helper.make_node('Add', ['with_col', 'inter_logits_out'], ['output']),
    ]
    return make_model(nodes, initializers, opset=11)


def task357_width_output_tensor(width):
    out = np.zeros((CH, H, W), dtype=np.float16)
    seed_row = 9
    seed_col = 0
    for r in range(10):
        for c in range(width):
            step = seed_row - r
            path_col = seed_col + bounce_col(step, width)
            out[1 if c == path_col else 8, r, c] = 1.0
    return out


def make_task357_bounce_path_model(payload):
    require_onnx()
    widths = list(range(2, 11))
    flat_size = CH * H * W
    out_size = CH * H * W

    W_det = np.zeros((len(widths), flat_size), dtype=np.float16)
    B_det = np.zeros((len(widths),), dtype=np.float16)
    for idx, width in enumerate(widths):
        # Top row is all zeros inside the visible width. The next column is inside for larger widths,
        # so penalizing all color channels there distinguishes each width except the max width.
        for c in range(width):
            W_det[idx, 0 * H * W + 0 * W + c] = 1.0
        if width < 10:
            for ch in range(CH):
                W_det[idx, ch * H * W + 0 * W + width] = -2.0
        B_det[idx] = -(width - 0.5)

    W_out = np.zeros((len(widths), out_size), dtype=np.float16)
    B_out = np.full((out_size,), -0.5, dtype=np.float16)
    for idx, width in enumerate(widths):
        W_out[idx, :] = task357_width_output_tensor(width).reshape(-1) * 4.0

    shape_1_ch_hw = numpy_helper.from_array(np.asarray([1, CH, H, W], dtype=np.int64), 'shape_1_ch_hw')
    initializers = [
        numpy_helper.from_array(W_det, 'W_width_det'), numpy_helper.from_array(B_det, 'B_width_det'),
        numpy_helper.from_array(W_out, 'W_width_out'), numpy_helper.from_array(B_out, 'B_width_out'),
        shape_1_ch_hw,
    ]
    nodes = [
        helper.make_node('Flatten', ['input'], ['flat_input'], axis=1),
        helper.make_node('Gemm', ['flat_input', 'W_width_det', 'B_width_det'], ['width_logits'], transB=1),
        helper.make_node('Relu', ['width_logits'], ['width_hits']),
        helper.make_node('Gemm', ['width_hits', 'W_width_out', 'B_width_out'], ['out_flat']),
        helper.make_node('Reshape', ['out_flat', 'shape_1_ch_hw'], ['output']),
    ]
    return make_model(nodes, initializers, opset=11)


def make_task176_periodic_completion_model(payload):
    require_onnx()
    W_base, B_base = make_base_identity_initializers()
    W_zero, B_zero = make_zero_mask_initializer()

    mask = np.zeros((1, 1, H, W), dtype=np.float16)
    for c in range(W):
        if c % 12 in (5, 6, 7):
            mask[0, 0, 0, c] = 1.0
        if c % 6 == 0:
            mask[0, 0, 1, c] = 1.0
        if c % 12 in (0, 1, 11):
            mask[0, 0, 2, c] = 1.0

    W_delta, B_delta = make_marker_delta_initializer(4)
    initializers = [
        W_base, B_base,
        W_zero, B_zero,
        numpy_helper.from_array(mask, 'periodic_mask'),
        W_delta, B_delta,
    ]
    nodes = [
        helper.make_node('Conv', ['input', 'W_base', 'B_base'], ['base_logits'], kernel_shape=[1, 1]),
        helper.make_node('Conv', ['input', 'W_zero_mask', 'B_zero_mask'], ['zero_logits'], kernel_shape=[1, 1]),
        helper.make_node('Relu', ['zero_logits'], ['zero_mask']),
        helper.make_node('Mul', ['zero_mask', 'periodic_mask'], ['marker_hits']),
        helper.make_node('Conv', ['marker_hits', 'W_marker_delta', 'B_marker_delta'], ['marker_logits'], kernel_shape=[1, 1]),
        helper.make_node('Add', ['base_logits', 'marker_logits'], ['output']),
    ]
    return make_model(nodes, initializers, opset=10)


def train_family_task(task):
    payload, info = fit_task050_line_connect(task)
    if info.get('ok'):
        return make_task050_line_connect_model(payload), info

    payload, info = fit_task126_u_bottom(task)
    if info.get('ok'):
        return make_task126_u_bottom_model(payload), info

    payload, info = fit_task299_cross_extend(task)
    if info.get('ok'):
        return make_task299_cross_extend_model(payload), info

    payload, info = fit_task357_bounce_path(task)
    if info.get('ok'):
        return make_task357_bounce_path_model(payload), info

    payload, info = fit_task176_periodic_completion(task)
    if info.get('ok'):
        return make_task176_periodic_completion_model(payload), info

    return None, {
        'ok': False,
        'trainer': 'identity_fallback_export',
        'model_version': MODEL_VERSION,
        'reason': 'no v4 exportable semantic task rule matched; identity fallback used by build cell',
    }





# --- Correctness-first registry ---------------------------------------------------
# Exportable predictors have an ONNX builder in train_family_task. Simulator-only
# predictors are visible-correct hypotheses that still need ONNX export work.
EXPORTABLE_PREDICTORS = {
    'task050': ('task050_line_connect_cnn', simulate_task050_line_connect),
    'task126': ('task126_u_bottom_marker_cnn', simulate_task126_u_bottom),
    'task176': ('task176_periodic_completion_cnn', simulate_task176_periodic_completion),
    'task299': ('task299_cross_extend_cnn', simulate_task299_cross_extend),
    'task357': ('task357_bounce_path_gemm', simulate_task357_bounce_path),
}

SIMULATOR_ONLY_PREDICTORS = {
    'task371': ('task371_midpoint_plus_simulator_only', simulate_task371_midpoint_plus),
}


def evaluate_predictor_on_task(task, predictor):
    right = 0
    total = 0
    first_wrong = None
    split_counts = {}
    for split in ['train', 'test', 'arc-gen']:
        split_right = 0
        split_total = 0
        for idx, ex in enumerate(task.get(split, [])):
            split_total += 1
            total += 1
            pred = predictor(ex['input'])
            if pred == ex['output']:
                split_right += 1
                right += 1
            elif first_wrong is None:
                first_wrong = f'{split}[{idx}]'
        split_counts[f'{split}_right'] = split_right
        split_counts[f'{split}_total'] = split_total
    return {
        'right': right,
        'total': total,
        'accuracy': right / total if total else None,
        'first_wrong': first_wrong,
        **split_counts,
    }


def identity_grid_predictor(input_grid):
    return [list(row) for row in input_grid]



In [8]:
# Correctness-first 41-task matrix.
# The target is to move every task from no_predictor/identity to visible_correct with a semantic predictor.
if 'task_ids' not in globals():
    task_map = load_task_type_map()
    family_df = task_map[task_map.primary_family == FAMILY].copy()
    family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
    nonlocal_1color_df = family_df[
        family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
        & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
        & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
    ].copy().sort_values('task_id').reset_index(drop=True)
    task_ids = nonlocal_1color_df['task_id'].tolist()

correctness_rows = []
for task_id in task_ids:
    task = load_task(DATA_DIR, task_id)
    status = 'no_predictor'
    rule_name = None
    export_status = 'none'
    predictor = None

    if task_id in EXPORTABLE_PREDICTORS:
        rule_name, predictor = EXPORTABLE_PREDICTORS[task_id]
        export_status = 'exportable'
        status = 'has_predictor'
    elif task_id in SIMULATOR_ONLY_PREDICTORS:
        rule_name, predictor = SIMULATOR_ONLY_PREDICTORS[task_id]
        export_status = 'simulator_only'
        status = 'has_predictor'
    else:
        rule_name = 'identity_baseline'
        predictor = identity_grid_predictor
        export_status = 'identity_fallback'
        status = 'identity_baseline'

    summary = evaluate_predictor_on_task(task, predictor)
    if status == 'has_predictor':
        status = 'visible_correct' if summary['right'] == summary['total'] else 'partial'
    elif status == 'identity_baseline' and summary['right'] == summary['total']:
        status = 'identity_correct'

    correctness_rows.append({
        'task_id': task_id,
        'status': status,
        'rule_name': rule_name,
        'export_status': export_status,
        'right': summary['right'],
        'total': summary['total'],
        'accuracy': summary['accuracy'],
        'first_wrong': summary['first_wrong'],
        'train_right': summary['train_right'],
        'train_total': summary['train_total'],
        'test_right': summary['test_right'],
        'test_total': summary['test_total'],
        'arc_gen_right': summary['arc-gen_right'],
        'arc_gen_total': summary['arc-gen_total'],
    })

correctness_df = pd.DataFrame(correctness_rows)
display(correctness_df)
display(correctness_df['status'].value_counts().rename_axis('status').reset_index(name='count'))
display(correctness_df['export_status'].value_counts().rename_axis('export_status').reset_index(name='count'))

unsolved_df = correctness_df[~correctness_df['status'].isin(['visible_correct', 'identity_correct'])].copy()
print('visible-correct tasks:', int((correctness_df['status'] == 'visible_correct').sum()), '/', len(correctness_df))
print('unsolved tasks:', len(unsolved_df))
display(unsolved_df[['task_id', 'status', 'right', 'total', 'accuracy', 'first_wrong']])


,task_id,status,rule_name,export_status,right,total,accuracy,first_wrong,train_right,train_total,test_right,test_total,arc_gen_right,arc_gen_total
0,task002,identity_baseline,identity_baseline,identity_fallback,0,268,0.000000,train[0],0,5,0,1,0,262
1,task027,identity_baseline,identity_baseline,identity_fallback,0,265,0.000000,train[0],0,3,0,1,0,261
2,task042,identity_baseline,identity_baseline,identity_fallback,0,266,0.000000,train[0],0,3,0,1,0,262
3,task043,identity_baseline,identity_baseline,identity_fallback,0,266,0.000000,train[0],0,3,0,1,0,262
4,task047,identity_baseline,identity_baseline,identity_fallback,0,265,0.000000,train[0],0,2,0,1,0,262
5,task050,visible_correct,task050_line_connect_cnn,exportable,271,271,1.000000,None,8,8,1,1,262,262
6,task060,identity_baseline,identity_baseline,identity_fallback,0,265,0.000000,train[0],0,2,0,1,0,262
7,task063,identity_baseline,identity_baseline,identity_fallback,0,266,0.000000,train[0],0,3,0,1,0,262
8,task090,identity_baseline,identity_baseline,identity_fallback,0,267,0.000000,train[0],0,4,0,1,0,262
9,task102,identity_baseline,identity_baseline,identity_fallback,115,267,0.430712,train[0],1,4,0,1,114,262


,status,count
0,identity_baseline,35
1,visible_correct,6


,export_status,count
0,identity_fallback,35
1,exportable,5
2,simulator_only,1


visible-correct tasks: 6 / 41
unsolved tasks: 35


,task_id,status,right,total,accuracy,first_wrong
0,task002,identity_baseline,0,268,0.000000,train[0]
1,task027,identity_baseline,0,265,0.000000,train[0]
2,task042,identity_baseline,0,266,0.000000,train[0]
3,task043,identity_baseline,0,266,0.000000,train[0]
4,task047,identity_baseline,0,265,0.000000,train[0]
6,task060,identity_baseline,0,265,0.000000,train[0]
7,task063,identity_baseline,0,266,0.000000,train[0]
8,task090,identity_baseline,0,267,0.000000,train[0]
9,task102,identity_baseline,115,267,0.430712,train[0]
10,task105,identity_baseline,0,266,0.000000,train[0]


In [9]:
# Build one model file for every selected nonlocal_1color task.
# The build uses exportable semantic models where available and identity fallback otherwise.
import shutil

if 'task_ids' not in globals():
    task_map = load_task_type_map()
    family_df = task_map[task_map.primary_family == FAMILY].copy()
    family_df['parsed_new_output_colors'] = family_df['new_output_color_list'].apply(parse_color_list)
    nonlocal_1color_df = family_df[
        family_df['candidate_flags'].fillna('').str.contains('adds_new_color_preserves_input')
        & ~family_df['candidate_flags'].fillna('').str.contains('local_3x3_consistent')
        & family_df['parsed_new_output_colors'].apply(lambda colors: len(colors) == 1)
    ].copy().sort_values('task_id').reset_index(drop=True)
    task_ids = nonlocal_1color_df['task_id'].tolist()

# Clear stale models from earlier runs before creating this scoped zip.
for old_model_path in OUT_DIR.glob('task*.onnx'):
    old_model_path.unlink()

print('pre-build task ids:', task_ids)
rows, zip_path = build_family_submission(
    FAMILY,
    train_family_task,
    DATA_DIR,
    OUT_DIR,
    fallback_identity=True,
    validate=False,
    task_ids_override=task_ids,
)

result_df = pd.DataFrame(rows)
display(result_df)
saved_count = int(result_df.get('saved', pd.Series(dtype=bool)).sum()) if len(result_df) else 0
print('selected nonlocal_1color tasks:', len(task_ids))
print('models saved:', saved_count)
if len(result_df) and 'trainer' in result_df:
    display(result_df['trainer'].fillna('none').value_counts().rename_axis('trainer').reset_index(name='count'))

expected_names = {f'{task_id}.onnx' for task_id in task_ids}
actual_names = {path.name for path in OUT_DIR.glob('task*.onnx')}
print('missing models:', sorted(expected_names - actual_names))
print('extra models:', sorted(actual_names - expected_names))
assert not (expected_names - actual_names), 'missing scoped task models'
assert not (actual_names - expected_names), 'found stale or out-of-scope task models'

# Kaggle looks for /kaggle/working/submission.zip when submitting from a notebook.
submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
shutil.copy2(zip_path, submission_zip)
print('family zip:', zip_path)
print('kaggle submission zip:', submission_zip)

pre-build task ids: ['task002', 'task027', 'task042', 'task043', 'task047', 'task050', 'task060', 'task063', 'task090', 'task102', 'task105', 'task119', 'task126', 'task139', 'task162', 'task166', 'task176', 'task200', 'task219', 'task232', 'task246', 'task251', 'task255', 'task265', 'task273', 'task278', 'task299', 'task303', 'task323', 'task335', 'task336', 'task341', 'task348', 'task350', 'task357', 'task367', 'task371', 'task381', 'task387', 'task392', 'task397']


,task_id,saved,path,ok,trainer,model_version,reason,fallback,semantic_kind,visible_right,visible_total,estimated_params,estimated_static_memory_bytes
0,task002,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
1,task027,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
2,task042,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
3,task043,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
4,task047,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
5,task050,True,/kaggle/working/working_submission/fill_enclos...,True,task050_line_connect_cnn,fill-additive-nonlocal-1color-v0.5-correctness,None,NaN,connect_color8_pairs_row_or_column_with_3,271.0,271.0,1500.0,14400.0
6,task060,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
7,task063,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
8,task090,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN
9,task102,True,/kaggle/working/working_submission/fill_enclos...,False,identity_fallback_export,fill-additive-nonlocal-1color-v0.5-correctness,no v4 exportable semantic task rule matched; i...,identity,NaN,NaN,NaN,NaN,NaN


selected nonlocal_1color tasks: 41
models saved: 41


,trainer,count
0,identity_fallback_export,36
1,task050_line_connect_cnn,1
2,task126_u_bottom_marker_cnn,1
3,task176_periodic_completion_cnn,1
4,task299_cross_extend_cnn,1
5,task357_bounce_path_gemm,1


missing models: []
extra models: []
family zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [10]:
# Model/version manifest for this notebook run.
run_manifest = {
    'family': FAMILY,
    'subtype': SUBTYPE,
    'model_version': MODEL_VERSION,
    'task_count': len(task_ids),
    'task_ids': task_ids,
    'out_dir': str(OUT_DIR),
    'strategy': 'task050/task126/task299/task357 semantic models, identity export fallback',
}
run_manifest


{'family': 'fill_enclosed_regions',
 'subtype': 'nonlocal_1color',
 'model_version': 'fill-additive-nonlocal-1color-v0.5-correctness',
 'task_count': 41,
 'task_ids': ['task002',
  'task027',
  'task042',
  'task043',
  'task047',
  'task050',
  'task060',
  'task063',
  'task090',
  'task102',
  'task105',
  'task119',
  'task126',
  'task139',
  'task162',
  'task166',
  'task176',
  'task200',
  'task219',
  'task232',
  'task246',
  'task251',
  'task255',
  'task265',
  'task273',
  'task278',
  'task299',
  'task303',
  'task323',
  'task335',
  'task336',
  'task341',
  'task348',
  'task350',
  'task357',
  'task367',
  'task371',
  'task381',
  'task387',
  'task392',
  'task397'],
 'out_dir': '/kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color',
 'strategy': 'task050/task126/task299/task357 semantic models, identity export fallback'}

In [11]:
# Optional: validate saved ONNX models on visible examples.
# This can be slow for large families and requires onnxruntime.
validate_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    summary = visible_validation_summary(row['path'], task)
    validate_rows.append({
        'task_id': row['task_id'],
        'right': summary['right'],
        'wrong': summary['wrong'],
    })

pd.DataFrame(validate_rows)

,task_id,right,wrong
0,task002,0,268
1,task027,0,265
2,task042,0,266
3,task043,0,266
4,task047,0,265
5,task050,271,0
6,task060,0,265
7,task063,0,266
8,task090,0,267
9,task102,115,152


In [12]:
# Submission helper.
# Rebuild the zip from this scoped OUT_DIR only.
submission_zip = create_submission_zip(OUT_DIR)
kaggle_submission_zip = Path('/kaggle/working/submission.zip') if Path('/kaggle/working').exists() else Path.cwd() / 'submission.zip'
import shutil
shutil.copy2(submission_zip, kaggle_submission_zip)
print('scoped zip:', submission_zip)
print('kaggle submission zip:', kaggle_submission_zip)


scoped zip: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/submission.zip
kaggle submission zip: /kaggle/working/submission.zip


In [13]:
# Architecture, performance, and memory report for saved models.
# This cell expects train_family_task to save one or more ONNX models.
# It reports the metrics the competition cares about: file size, parameter
# count, and memory profile, plus train/test/arc-gen exact-match performance.

report_rows = []
for row in rows:
    if not row.get('saved'):
        continue
    task = load_task(DATA_DIR, row['task_id'])
    try:
        report = model_report(row['path'], task=task)
        arch = report['architecture']
        mem = report['memory_profile']
        perf = report['performance']
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'file_size_bytes': arch.get('file_size_bytes'),
            'params': arch.get('params'),
            'nodes': arch.get('nodes'),
            'op_counts': json.dumps(arch.get('op_counts', {}), sort_keys=True),
            'static_memory_bytes': mem.get('static_memory_bytes'),
            'runtime_memory_bytes': mem.get('runtime_memory_bytes'),
            'train_right': perf['train']['right'],
            'train_total': perf['train']['total'],
            'train_accuracy': perf['train']['accuracy'],
            'test_right': perf['test']['right'],
            'test_total': perf['test']['total'],
            'test_accuracy': perf['test']['accuracy'],
            'arc_gen_right': perf['arc_gen']['right'],
            'arc_gen_total': perf['arc_gen']['total'],
            'arc_gen_accuracy': perf['arc_gen']['accuracy'],
            'visible_right': perf['visible_all']['right'],
            'visible_total': perf['visible_all']['total'],
            'visible_accuracy': perf['visible_all']['accuracy'],
        })
    except Exception as exc:
        report_rows.append({
            'task_id': row['task_id'],
            'model_version': MODEL_VERSION,
            'profile_error': repr(exc),
        })

profile_df = pd.DataFrame(report_rows)
display(profile_df)

,task_id,model_version,file_size_bytes,params,nodes,op_counts,static_memory_bytes,runtime_memory_bytes,train_right,train_total,train_accuracy,test_right,test_total,test_accuracy,arc_gen_right,arc_gen_total,arc_gen_accuracy,visible_right,visible_total,visible_accuracy
0,task002,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,5,0.00,0,1,0.0,0,262,0.000000,0,268,0.000000
1,task027,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,261,0.000000,0,265,0.000000
2,task042,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.000000
3,task043,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.000000
4,task047,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,2,0.00,0,1,0.0,0,262,0.000000,0,265,0.000000
5,task050,fill-additive-nonlocal-1color-v0.5-correctness,4145,1345,19,"{""Add"": 2, ""Cast"": 2, ""Conv"": 7, ""Mul"": 3, ""Re...",97200,99890,8,8,1.00,1,1,1.0,262,262,1.000000,271,271,1.000000
6,task060,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,2,0.00,0,1,0.0,0,262,0.000000,0,265,0.000000
7,task063,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,3,0.00,0,1,0.0,0,262,0.000000,0,266,0.000000
8,task090,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,0,4,0.00,0,1,0.0,0,262,0.000000,0,267,0.000000
9,task102,fill-additive-nonlocal-1color-v0.5-correctness,471,110,3,"{""Cast"": 2, ""Conv"": 1}",36000,36220,1,4,0.25,0,1,0.0,114,262,0.435115,115,267,0.430712


In [14]:
# Persist run metadata next to the generated models.
if 'profile_df' in globals() and len(profile_df):
    profile_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_profile.csv'
    profile_df.to_csv(profile_path, index=False)
    print('wrote profile:', profile_path)

manifest_path = OUT_DIR / f'{FAMILY}_{MODEL_VERSION}_manifest.json'
with open(manifest_path, 'w') as f:
    json.dump(run_manifest, f, indent=2)
print('wrote manifest:', manifest_path)

wrote profile: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.5-correctness_profile.csv
wrote manifest: /kaggle/working/working_submission/fill_enclosed_regions_nonlocal_1color/fill_enclosed_regions_fill-additive-nonlocal-1color-v0.5-correctness_manifest.json
